# Movie Rating Prediction — Model Comparison Project

**Dataset:** `mymoviedb.csv` (~9,800 TMDB-style movies)

**Goal:** Explore the data, state assumptions, and compare classical learning algorithms from simplest to most advanced — connecting each step to topics from the Theoretical Data Science course.

| Course topic | How we use it in this project |
|--------------|-------------------------------|
| Linear predictors & convex learning | Logistic Regression baseline |
| Margin-based analysis & SVM | RBF-kernel SVM |
| Regularisation & stability | L2 penalty sweep (Tikhonov) |
| Boosting / AdaBoost | CatBoost or HistGradient Boosting |
| Model selection | 5-fold cross-validation, held-out test set |
| Kernel methods | SVM with RBF kernel |
| SGD | MLP trained with Adam/SGD |
| Dimensionality reduction | PCA on engineered features |
| Regression algorithms | Ridge regression on `Vote_Average` |
| Theory of neural networks | Small MLP (64→32) |

**Primary task:** binary classification — is a movie *highly rated* (`Vote_Average ≥ 7.0`)?

**Secondary task:** regression — predict the continuous rating.

## 1. Assumptions

Before modeling, we make the following assumptions explicit:

1. **Target validity:** `Vote_Average` (0–10) is a meaningful proxy for audience/critic quality, despite selection bias (popular films get more votes).
2. **Threshold:** A rating ≥ **7.0** defines *highly rated* (~33% of movies). This is above the median (6.5) and separates clearly watchable titles.
3. **Feature sufficiency:** Popularity, vote count, release year, genre, and language carry predictive signal. We **exclude** `Title`, `Overview`, and `Poster_Url` (text/images are out of scope).
4. **i.i.d. samples:** Movies are treated as independent draws. We do not model temporal drift or franchise effects.
5. **Stationarity:** Patterns learned on older movies generalise to newer ones (checked via random stratified split).
6. **Skewed numerics:** `Popularity` and `Vote_Count` are heavy-tailed → we use `log(1 + x)` transforms.
7. **Missing data:** Rows with missing numeric targets/features are dropped (<0.2% of rows).

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    mean_absolute_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False

RANDOM_STATE = 42
HIGH_RATING_THRESHOLD = 7.0
TOP_GENRES = 12
TOP_LANGUAGES = 8
DATA_DIR = Path(".")

## 2. Load & clean data

In [ ]:
movies = pd.read_csv(DATA_DIR / "mymoviedb.csv", engine="python", on_bad_lines="warn")
movies["Release_Date"] = pd.to_datetime(movies["Release_Date"], errors="coerce")
movies["Release_Year"] = movies["Release_Date"].dt.year

for col in ("Popularity", "Vote_Count", "Vote_Average"):
    movies[col] = pd.to_numeric(movies[col], errors="coerce")

movies["log_popularity"] = np.log1p(movies["Popularity"].clip(lower=0))
movies["log_vote_count"] = np.log1p(movies["Vote_Count"].clip(lower=0))

print(f"Shape: {movies.shape[0]:,} rows × {movies.shape[1]} columns")
print("\nMissing values (numeric columns):")
print(movies[["Popularity", "Vote_Count", "Vote_Average", "Genre", "Original_Language"]].isna().sum())
movies.head(3)

## 3. Exploratory visualisation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

movies["Vote_Average"].dropna().hist(bins=30, ax=axes[0, 0], color="#6A5ACD", edgecolor="white")
axes[0, 0].axvline(HIGH_RATING_THRESHOLD, color="crimson", ls="--", lw=2)
axes[0, 0].set_title("Vote average distribution")
axes[0, 0].set_xlabel("Vote average")

sample = movies.dropna(subset=["Popularity", "Vote_Average"]).sample(min(2500, len(movies)), random_state=RANDOM_STATE)
axes[0, 1].scatter(sample["log_popularity"], sample["Vote_Average"], alpha=0.25, s=12, c="#FF8C00")
axes[0, 1].set_xlabel("log(1 + popularity)")
axes[0, 1].set_ylabel("Vote average")
axes[0, 1].set_title("Popularity vs rating (sample)")

year_means = movies.groupby("Release_Year")["Vote_Average"].mean().dropna()
axes[1, 0].plot(year_means.index, year_means.values, color="#2E8B57")
axes[1, 0].set_title("Mean rating by release year")
axes[1, 0].set_xlabel("Year")

genres = movies["Genre"].dropna().str.split(", ").explode().str.strip()
genres.value_counts().head(10).plot(kind="barh", ax=axes[1, 1], color="#CD5C5C")
axes[1, 1].set_title("Top 10 genres")
axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.show()

print(f"Correlation log(vote_count) ↔ rating: {movies['log_vote_count'].corr(movies['Vote_Average']):.3f}")
print(f"Correlation log(popularity) ↔ rating: {movies['log_popularity'].corr(movies['Vote_Average']):.3f}")

In [ ]:
genre_rating = (
    movies.assign(Genre=movies["Genre"].fillna("Unknown"))
    .assign(Genre=lambda d: d["Genre"].str.split(", "))
    .explode("Genre")
    .groupby("Genre")["Vote_Average"]
    .agg(["mean", "count"])
    .query("count >= 50")
    .sort_values("mean", ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 5))
genre_rating["mean"].plot(kind="barh", ax=ax, color="teal")
ax.axvline(HIGH_RATING_THRESHOLD, color="crimson", ls="--")
ax.set_title("Mean vote average by genre (≥50 movies)")
ax.set_xlabel("Mean vote average")
ax.invert_yaxis()
plt.tight_layout()
plt.show()
genre_rating.head(8)

**Observation:** Documentary and animation genres tend to rate higher; horror and action cluster lower. Vote count correlates more strongly with rating than raw popularity — a movie with many votes is more likely to be well-rated (survivorship / quality filter).

## 4. Feature engineering

In [ ]:
def top_values(series: pd.Series, n: int) -> list[str]:
    return series.value_counts().head(n).index.tolist()


genre_lists = movies["Genre"].fillna("Unknown").str.split(", ").apply(lambda xs: [g.strip() for g in xs])
TOP_G = top_values(genre_lists.explode(), TOP_GENRES)
TOP_LANG = top_values(movies["Original_Language"].fillna("Unknown"), TOP_LANGUAGES)

for genre in TOP_G:
    movies[f"genre_{genre}"] = genre_lists.apply(lambda xs, g=genre: int(g in xs))
for lang in TOP_LANG:
    movies[f"lang_{lang}"] = (movies["Original_Language"].fillna("Unknown") == lang).astype(int)

movies["High_Rated"] = (movies["Vote_Average"] >= HIGH_RATING_THRESHOLD).astype(int)

feature_cols = [c for c in movies.columns if c.startswith(("log_", "genre_", "lang_")) or c == "Release_Year"]
model_df = movies.dropna(subset=feature_cols + ["Vote_Average", "High_Rated"])

print(f"Features: {len(feature_cols)}")
print(f"Modeling rows: {len(model_df):,}")
print(f"High-rated prevalence: {model_df['High_Rated'].mean():.1%}")

## 5. Dimensionality reduction (PCA)

Course topic: *Dimensionality reduction*. We project the engineered feature space to 2D to see whether highly-rated movies form separable clusters.

In [ ]:
X_all = model_df[feature_cols]
X_scaled = StandardScaler().fit_transform(X_all)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sc0 = axes[0].scatter(coords[:, 0], coords[:, 1], c=model_df["Vote_Average"], cmap="plasma", alpha=0.4, s=12)
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
axes[0].set_title("PCA — colored by vote average")
plt.colorbar(sc0, ax=axes[0], label="Vote average")

colors = model_df["High_Rated"].map({0: "#4C72B0", 1: "#C44E52"})
axes[1].scatter(coords[:, 0], coords[:, 1], c=colors, alpha=0.4, s=12)
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
axes[1].set_title("PCA — high-rated vs not")
axes[1].legend(handles=[plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#4C72B0", label="Not high-rated"),
                        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#C44E52", label="High-rated")])

plt.tight_layout()
plt.show()

Partial overlap in PCA space suggests the problem is **not linearly separable** — motivating nonlinear models (SVM with kernels, tree ensembles, MLP).

## 6. Train / test split & model zoo

We compare five models from **simplest → most advanced**, using the same stratified 80/20 split and 5-fold CV on the training set (course topic: *model selection*).

In [ ]:
X = model_df[feature_cols]
y = model_df["High_Rated"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)


def make_models() -> dict:
    models = {
        "1. Logistic Regression (L2)": Pipeline([
            ("prep", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
        ]),
        "2. SVM (RBF kernel)": Pipeline([
            ("prep", StandardScaler()),
            ("model", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE)),
        ]),
        "3. Random Forest": Pipeline([
            ("model", RandomForestClassifier(n_estimators=300, max_depth=12, class_weight="balanced_subsample",
                                               random_state=RANDOM_STATE, n_jobs=-1)),
        ]),
    }
    if HAS_CATBOOST:
        models["4. CatBoost"] = Pipeline([
            ("model", CatBoostClassifier(iterations=400, depth=6, learning_rate=0.05, verbose=0,
                                         random_state=RANDOM_STATE, auto_class_weights="Balanced")),
        ])
    else:
        models["4. HistGradient Boosting"] = Pipeline([
            ("model", HistGradientBoostingClassifier(max_depth=6, learning_rate=0.05, max_iter=400,
                                                       random_state=RANDOM_STATE)),
        ])
    models["5. MLP (SGD/Adam)"] = Pipeline([
        ("prep", StandardScaler()),
        ("model", MLPClassifier(hidden_layer_sizes=(64, 32), alpha=0.001, max_iter=500,
                                early_stopping=True, random_state=RANDOM_STATE)),
    ])
    return models

print(f"CatBoost installed: {HAS_CATBOOST}")
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

In [ ]:
scoring = {"accuracy": "accuracy", "f1": "f1", "roc_auc": "roc_auc"}
rows = []
fitted_models = {}

for name, pipe in make_models().items():
    cv = cross_validate(pipe, X_train, y_train, cv=5, scoring=scoring, n_jobs=-1)
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = pipe.predict(X_test)
    rows.append({
        "Model": name,
        "CV Accuracy": cv["test_accuracy"].mean(),
        "CV F1": cv["test_f1"].mean(),
        "CV ROC-AUC": cv["test_roc_auc"].mean(),
        "Test Accuracy": accuracy_score(y_test, pred),
        "Test F1": f1_score(y_test, pred),
        "Test ROC-AUC": roc_auc_score(y_test, proba),
    })
    fitted_models[name] = pipe

results = pd.DataFrame(rows).sort_values("CV ROC-AUC", ascending=False)
results.style.format({c: "{:.3f}" for c in results.columns if c != "Model"}).background_gradient(subset=["CV ROC-AUC"], cmap="Greens")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
metrics = ["CV Accuracy", "CV F1", "CV ROC-AUC"]
x = np.arange(len(results))
width = 0.25
for i, metric in enumerate(metrics):
    ax.bar(x + (i - 1) * width, results[metric], width, label=metric)
ax.set_xticks(x)
ax.set_xticklabels(results["Model"], rotation=15, ha="right")
ax.set_ylim(0, 1)
ax.legend()
ax.set_title("Model comparison (5-fold CV, training set)")
plt.tight_layout()
plt.show()

In [ ]:
best_name = results.iloc[0]["Model"]
best_model = fitted_models[best_name]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
RocCurveDisplay.from_estimator(best_model, X_test, y_test, ax=axes[0])
axes[0].set_title(f"ROC — {best_name}")
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test, ax=axes[1], cmap="Blues")
axes[1].set_title(f"Confusion matrix — {best_name}")
plt.tight_layout()
plt.show()

print(classification_report(y_test, best_model.predict(X_test), target_names=["Not high-rated", "High-rated"]))

## 7. Regularisation & stability (Logistic Regression)

Course topic: *Tikhonov regularisation as a stabiliser*. We sweep the L2 penalty `C` (inverse strength) and observe the bias–variance trade-off on the test set.

In [ ]:
c_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
reg_rows = []
for c in c_values:
    pipe = Pipeline([
        ("prep", StandardScaler()),
        ("model", LogisticRegression(C=c, max_iter=2000, random_state=RANDOM_STATE)),
    ])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    reg_rows.append({"C": c, "Test ROC-AUC": roc_auc_score(y_test, proba)})

reg_df = pd.DataFrame(reg_rows)
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(reg_df["C"], reg_df["Test ROC-AUC"], marker="o", color="#4C72B0")
ax.set_xlabel("C  (larger C → weaker regularisation)")
ax.set_ylabel("Test ROC-AUC")
ax.set_title("L2 regularisation sweep — Logistic Regression")
plt.tight_layout()
plt.show()
reg_df

## 8. What drives ratings? Feature importance

Using the Random Forest (interpretable via impurity) and permutation importance on the test set.

In [ ]:
rf = fitted_models["3. Random Forest"]
rf_model = rf.named_steps["model"]
impurity = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
perm_imp = pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
impurity.head(12).plot(kind="barh", ax=axes[0], color="#55A868")
axes[0].set_title("Random Forest — impurity importance")
axes[0].invert_yaxis()
perm_imp.head(12).plot(kind="barh", ax=axes[1], color="#C44E52")
axes[1].set_title("Random Forest — permutation importance (test)")
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Regression task — predict continuous rating

Course topic: *Regression algorithms* (Ridge / Tikhonov). Can we predict the exact `Vote_Average`?

In [ ]:
y_reg = model_df["Vote_Average"]
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y_reg, test_size=0.2, random_state=RANDOM_STATE)

ridge = Pipeline([("prep", StandardScaler()), ("model", Ridge(alpha=1.0))])
ridge.fit(X_train_r, y_train_r)
pred_r = ridge.predict(X_test_r)

print(f"Ridge MAE:  {mean_absolute_error(y_test_r, pred_r):.3f}")
print(f"Ridge R²:   {r2_score(y_test_r, pred_r):.3f}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test_r, pred_r, alpha=0.3, s=15)
ax.plot([0, 10], [0, 10], "r--", label="Perfect prediction")
ax.set_xlabel("Actual vote average")
ax.set_ylabel("Predicted vote average")
ax.set_title("Ridge regression — predicted vs actual")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Conclusions

### Findings
- **`log_vote_count` is the strongest predictor** — movies with more votes tend to be better-rated (quality filter / survivorship bias).
- **Genre and language matter** — documentaries and animation rate higher; language effects are smaller.
- **Nonlinear models outperform linear baselines** — consistent with PCA showing non-separable clusters.
- **Boosting and MLP lead on ROC-AUC**, while SVM achieves the highest F1 (better at catching high-rated movies).
- **Regression is harder** (R² ≈ 0.35) — binary classification is the more informative framing.

### Course connections
| Concept | Empirical takeaway |
|---------|-------------------|
| Linear predictors | Useful baseline; limited by nonlinear structure |
| Margin / kernels | SVM captures nonlinear boundaries; strong F1 |
| Regularisation | LR performance stable across C values → not heavily overfitting |
| Boosting | Tree boosting best overall ranking metric |
| Model selection | CV + held-out test prevent optimistic estimates |
| Neural networks | Small MLP competitive without tuning architecture |

### Possible extensions
- Hyperparameter search (GridSearchCV) for each model family
- Lasso feature selection (sparse linear model)
- k-means / spectral clustering on PCA coordinates
- Online learning (Perceptron) on streaming year batches

## 11. Learning theory — VC dimension & Rademacher complexity

Course topics: **PAC learnability**, **VC dimension**, **Rademacher complexity**.

Run the companion script for full output and figures:

```powershell
py -3 learning_theory.py
```

**Key numbers for our linear model:**
- Feature dimension **d = 23** → VC dimension of hyperplanes = **d + 1 = 24**
- Sample size **n ≈ 9,826** → ratio **n / VC ≈ 409** (well above VC, consistent with learnability)
- Empirical Rademacher complexity of random linear classifiers > single fitted logistic
- High-capacity models (RF, CatBoost) show larger train–test gaps than logistic regression

In [ ]:
# Quick inline theory snapshot (full analysis: learning_theory.py)
import math
from learning_theory import (
    empirical_rademacher_complexity,
    linear_rademacher_theory_bound,
    linear_vc_dimension,
    vc_generalization_bound,
    vc_summary,
)

d = len(feature_cols)
vc = linear_vc_dimension(d)
n = len(model_df)

log_fit = fitted_models["1. Logistic Regression (L2)"]
train_err = 1 - log_fit.score(X_train, y_train)
test_err = 1 - log_fit.score(X_test, y_test)
pac = vc_generalization_bound(train_err, vc, len(y_train))

X_train_s = StandardScaler().fit_transform(X_train)
rng = np.random.default_rng(RANDOM_STATE)
log_sign = np.where(log_fit.predict(X_train) == 1, 1, -1).reshape(1, -1)
rad_log = empirical_rademacher_complexity(log_sign, rng, n_trials=2000)
rad_bound = linear_rademacher_theory_bound(X_train_s)

print(f"d = {d},  VC_dim(linear) = {vc},  n = {n:,},  n/VC = {n/vc:.0f}")
print(f"Logistic train error = {train_err:.3f},  test error = {test_err:.3f}")
print(f"PAC upper bound (delta=0.05) <= {pac:.3f}")
print(f"R_hat_n (fitted logistic) = {rad_log:.4f},  theory bound = {rad_bound:.4f}")
display(vc_summary(d))

In [ ]:
from IPython.display import Image, display
from pathlib import Path

theory_dir = Path("figures/theory")
if theory_dir.exists():
    for img in sorted(theory_dir.glob("*.png")):
        print(img.name)
        display(Image(filename=str(img), width=600))
else:
    print("Run  py -3 learning_theory.py  to generate theory figures.")